# 第 2 周练习 —— The BookFather 书店顾问

## 练习目标

用 **CSV 书目数据 + OpenAI Chat Completions（流式）+ Gradio** 做一个「按品类与预算荐书」的小应用：

- **输入**：书籍品类（Category）、最高预算（Max price）
- **处理**：在本地 DataFrame 里筛选/排序，把 Top 结果交给模型组织话术
- **输出**：Gradio 界面里流式显示顾问回复（Markdown）

## 和本课 Week 2 的关系

| 概念 | 本练习里你会看到 |
|------|------------------|
| 结构化数据过滤 | `pandas` 筛选 `Book_category` / `Price` |
| Tool-ish 工作流 | 先查表，再把 JSON 结果塞进 user prompt |
| 流式输出 `stream=True` | `openai_stream` 里边收边 `yield` |
| Gradio UI | `gr.Interface` 把函数挂成可点的网页 |

## 怎么跑

1. 先下载 CSV，放到笔记本同目录，文件名保持 `books_scraped.csv`
2. 准备好 `.env` 里的 `OPENAI_API_KEY`
3. 从上到下运行单元格，最后 `view.launch()` 打开界面

运行本项目前请先下载 CSV 文件：

https://drive.google.com/file/d/1WByAMpHJXZ5oTY120KEvBGRxTZzXoNQC/view?usp=sharing


In [ ]:
# ========== 导入：数据处理 + 环境变量 + OpenAI + 笔记本展示 ==========

# 导入 pandas：用 DataFrame 读写 CSV、筛选书目
import pandas as pd

# 导入标准库 os：从环境变量读取 API Key
import os
# 导入标准库 json：本练习后续可能用到 JSON 序列化（与 to_json 配合的思路）
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具（Markdown / display / update_display）
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI


In [ ]:
# ========== 环境检查：加载 .env 并粗测 OPENAI_API_KEY 形态 ==========

# 从 requests 包导入 api 子模块（原代码如此；本格主要做密钥检查）
from requests import api


# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv (override = True)
# 读取 OpenAI API Key（名字必须是 OPENAI_API_KEY，和常见课程配置一致）
api_key = os.getenv('OPENAI_API_KEY')
# 粗测：存在且以 sk-proj- 开头则打印「看起来不错」（排错文案保持英文原样）
if api_key and api_key.startswith("sk-proj-") : 
  print ("API Key looks good so far")
else : 
  print ("There might be a proble, with your API key ! ")
  



In [ ]:
# ========== 模型与客户端：选定云端模型并创建 OpenAI() ==========

# 模型 id 集中存放；字符串必须保持原样（计费/可用性取决于这个名字）
Model = 'gpt-5-nano'
# 创建默认 OpenAI 客户端（密钥来自环境变量）
openai = OpenAI()


In [ ]:
# ========== 读数据：把抓取好的书目 CSV 载入 DataFrame ==========

# 读取同目录下的 books_scraped.csv；路径字符串保持原样
book_scraped = pd.read_csv ("books_scraped.csv")
# 单元格末尾写变量名：在 Jupyter 里预览表格
book_scraped


In [ ]:
# ========== 品类列表：去重后供 Gradio 下拉框使用 ==========

# 取出 Book_category 列 → 去重 → 转成 Python list
Book_category = book_scraped["Book_category"].drop_duplicates().tolist()
# 预览有哪些品类可选
Book_category


In [ ]:
# ========== 清洗：丢掉 Stock 列（库存字段本练习不用） ==========

# axis=1 表示按列删除；inplace=True 直接改原 DataFrame
book_scraped.drop("Stock" , axis= 1 , inplace= True)
# 再预览一次清洗后的表
book_scraped


In [ ]:
# ========== 核心检索：按品类 + 最高价过滤，返回 Top5 或「超预算建议」 ==========

def smart_search (store_data ,category_name = None , max_price = None ) : 
   # 先 copy，避免筛选时误改调用方传入的原表
   result = store_data.copy()
   # 若给了品类名：不区分大小写地精确匹配 Book_category
   if category_name : 
      result = result[result["Book_category"].str.lower() == category_name.lower()]
   
   # 再按 Price <= max_price 过滤（预算上限）
   final_result = result [result ["Price"] <= max_price ]
   # 若预算内为空、且确实传了 max_price：给出该品类里略贵的 5 本建议
   if final_result.empty and max_price : 
      suggestions= result [ result ["Price"] > max_price].sort_values(by ="Price").head(5)
      # 提示文案保持英文原样（影响展示内容）
      message =  f"""
      The book category you want is more expensive than the budget or doesn't exist.
      there is some suggestions of category {category_name} 
       """ 
      # 返回元组：(说明文字, 建议表) —— 后面 return_book_info 会用 type==tuple 分支识别
      return message , suggestions 
   
   # 正常情况：按价格升序取最便宜的 5 本
   return final_result.sort_values(by ="Price").head(5)




In [ ]:
# ========== 冒烟测试：看 smart_search 返回的是 DataFrame 还是 tuple ==========

# 查询 Novels 且预算 50；用 type(...) 观察返回类型
type (smart_search(book_scraped , "Novels" , 50) )


In [ ]:
# ========== System Prompt：书店顾问人设与问诊流程（影响模型行为，勿改文案） ==========

# f-string 形式的 system 指令：身份、必须询问的两要素、语气；内容保持英文原样
system_prompt = f"""
 You are a professional and friendly AI Book Consultant for the "the Bookfather " bookstore.
 Objective: Your goal is to help customers find their perfect book based on a specific dataset.
  Workflow:

  Greeting: Start with a warm welcome.
  Information Gathering: You MUST ask the user for two things:
  Their preferred Book Category (e.g., Fiction, History, Poetry).
  Their Maximum Budget (Price).
  Guidance: Once the user provides this, inform them that you are searching the catalog.

  Tone: Helpful, concise, and professional. Do not hallucinate books that are not in the store's data
"""

In [ ]:
# ========== 把检索结果变成模型可读的 JSON（或原样返回超预算元组） ==========

def return_book_info ( category , max_price ) :
  # 基于全局书目再 copy 一份，交给 smart_search
  df = book_scraped.copy()
  result = smart_search (
    df,
    category_name= category ,
    max_price= max_price
  )
  # 若是 (message, suggestions) 元组：直接返回给上层处理
  if type (result) == tuple : 
    return result 
  else :
    # DataFrame → JSON 字符串（records：一行一本书的对象数组）
    json_str = result.to_json(orient="records", indent=2)
    return json_str





In [ ]:
# ========== 流式荐书：查表 → 拼 user prompt → stream=True 边生成边 yield ==========

def openai_stream (category , budget) : 
  # 先拿到书目 JSON（或超预算时的特殊返回值）
  books_json = return_book_info(category , budget)
  # user 侧任务说明：品类、预算、以及数据库 Top 匹配；prompt 正文保持英文
  prompt = f"""
The user is looking for books in the category: '{category}' 
with a maximum budget of: {budget}.

Based on our database, here are the top 5 matches in JSON format:
{books_json}

Please introduce these books to the user, highlighting their price and rating, and tell them why these are the best choices for them.
"""
  # messages：system 定顾问人设，user 放本次检索结果
  messages = [
    {"role" : "system" , "content" : system_prompt},
    {"role" : "user" , "content" : prompt} 
  ]

  # 创建流式补全；model 用上面的 Model 常量
  stream = openai.chat.completions.create( 
    model = Model,
    messages = messages, 
    stream = True 
  )
  # 累积已生成文本；每次 yield 当前全文（方便 Gradio 渐进刷新）
  result = ""
  for chunk in stream : 
      result += chunk.choices[0].delta.content or ""
      yield result


In [ ]:
# ========== 再测检索：打印 Novels / 预算 70 的筛选结果 ==========

# 直接 print DataFrame（或超预算时的元组），便于肉眼核对
print (smart_search(book_scraped ,"Novels" , 70))


In [ ]:
# ========== 导入 Gradio：后面用它做简单 Web UI ==========

# 导入 gradio，并习惯简写为 gr
import gradio as gr 


In [ ]:
# ========== Gradio 界面：下拉选品类 + 数字预算 → 流式 Markdown 回复 ==========

# 品类下拉框：选项来自 Book_category；默认 Poetry（标签文案保持英文）
category_selector = gr.Dropdown(Book_category , label = "select category" , value = "Poetry")
# 最高价数字输入；默认 70
max_price_input = gr.Number(label = "Max-value" , value = 70)
# 输出区：用 Markdown 组件展示顾问回复
message_output = gr.Markdown (label = "Response")

# 把 openai_stream 挂到 Interface：两个输入、一个输出；关闭 flagging
view = gr.Interface (
  fn = openai_stream ,
  title = "The BookFather ",
  inputs = [category_selector , max_price_input],
  outputs = [message_output],
  flagging_mode = "never"
)

# 启动本地 Gradio 应用（默认会打印访问 URL）
view.launch ()
